In [1]:
from dataclasses import dataclass
from langchain_core.messages import SystemMessage,HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.graph import StateGraph,START,END
from langgraph.runtime import Runtime
from loguru import logger
from langgraph.graph.message import MessagesState

from dotenv import  load_dotenv
load_dotenv(override=True)

model = ChatDeepSeek(
    model = "deepseek-flash",
    extra_body={
        "thinking":{
            "type":"disabled"
        }
    }
)

#1. 定义运行时的环境上下文
@dataclass
class UserContext:
    username:str
    membership_level:str

#2. 定义状态
class OverAllState(MessagesState):
    user_input:str
    output:str

#3. 定义节点
def llm_node(state:OverAllState,runtime:Runtime[UserContext]) -> OverAllState:
    #1. 获取环境上下文 判断当前的用户等级
    runtime_context = runtime.context

    if runtime_context:
        level = runtime_context.membership_level
        username = runtime_context.username
        logger.info(f"当前用户:{username},会员等级:{level}")

        if level == "VIP":
            system_prompt = f"你是高级客户助理,当前VIP用户是{username},请使用尊称'您',语气热情周到,回复末尾加上'VIP🏅服务'"
        else:
            system_prompt = f"你是普通客户助理,当前用户是{username},请友好简洁回复问题"

    else:
        system_prompt = f"你是普通客户助理,请友好简洁回复问题"

    user_input = state["user_input"]
    messages = state.get("messages",[])
    response = model.invoke([SystemMessage(content=system_prompt)]+ messages + [HumanMessage(content=user_input)]).content
    return {
        "messages" : messages,
        "output":response
    }

#4. 构建图
builder = StateGraph(state_schema=OverAllState,context_schema=UserContext)

builder.add_node("llm_node",llm_node)
builder.add_edge(START,"llm_node")
builder.add_edge("llm_node",END)

graph = builder.compile()

# ==========第一次调用: 传入VIP上下文用户====================
res = graph.invoke(
    {"user_input":"你好,帮我查一下最近有什么优惠活动"},
    context=UserContext(username="Alice",membership_level="VIP")
)
print(res)


2026-10-28 13:38:21.229 | INFO     | __main__:llm_node:40 - 当前用户:Alice,会员等级:VIP


{'messages': [], 'user_input': '你好,帮我查一下最近有什么优惠活动', 'output': 'Alice您好！非常高兴为您服务～关于最近的优惠活动，我马上为您查询。请问您比较关注哪一类优惠呢？比如购物折扣、会员专享还是积分兑换？这样我可以更精准地为您推荐哦～VIP🏅服务'}


In [2]:
# ==========第二次调用: 传入普通上下文用户====================
res1 = graph.invoke(
    {"user_input":"你好,帮我查一下最近有什么优惠活动"},
    context=UserContext(username="Alice",membership_level="普通用户")
)
print(res1)

2026-10-28 13:38:57.985 | INFO     | __main__:llm_node:40 - 当前用户:Alice,会员等级:普通用户


{'messages': [], 'user_input': '你好,帮我查一下最近有什么优惠活动', 'output': '你好，Alice！目前具体的优惠活动我这边暂时查不到实时信息，建议你登录官网或APP查看“活动专区”，也可以留意短信和邮件通知。如果你想了解某类产品（比如会员、套餐）的优惠，我可以帮你进一步说明～'}
